# 4단계 — BERT 파인튜닝 (KR-ELECTRA)

한국어 사전학습 모델 `snunlp/KR-ELECTRA-discriminator`를 스팸 분류에 맞게 파인튜닝합니다.

구조:
```
[CLS] 제목 [SEP] 본문 [SEP]
        ↓
   BERT Encoder
        ↓
  [CLS] 벡터 (768)
        ↓
  Dropout(0.3)
        ↓
  Linear(768→2) → Softmax → [ham 확률, spam 확률]
```

> **주의:** GPU가 있으면 훨씬 빠릅니다. CPU만 있는 경우 에폭당 수십 분 소요될 수 있습니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
print(f'PyTorch 버전: {torch.__version__}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 데이터 로드 & 전처리

In [ ]:
from src.data.preprocessor import Preprocessor
from src.data.loader import load_dataset

pre = Preprocessor(use_konlpy=False)

data = load_dataset(
    csv_path='../data/korean_spam_dataset.csv',
    preprocessor=pre,
    train_ratio=0.8,
    val_ratio=0.1,
    random_state=42,
)

## 토크나이저 확인

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'snunlp/KR-ELECTRA-discriminator'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = '무료 대출 1억 지금 바로 신청하세요'
tokens = tokenizer(sample, return_tensors='pt', max_length=64, truncation=True)
print('토큰 ID:', tokens['input_ids'][0].tolist())
print('디코딩:', tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))

## 모델 학습

> CPU 환경에서 빠른 테스트를 위해 `epochs=1`, 데이터 일부만 사용합니다.
> 실제 학습 시에는 `python -m src.train --model bert`를 사용하세요.

In [ ]:
from src.models.bert_model import BERTSpamClassifier

# CPU 빠른 테스트: 데이터 200개, 1에폭
QUICK_TEST = not torch.cuda.is_available()

if QUICK_TEST:
    print('[안내] CPU 환경 감지 — 소규모 데이터(200개)로 1에폭 테스트 실행')
    X_tr, y_tr = data.X_train[:200], data.y_train[:200]
    X_v,  y_v  = data.X_val[:50],   data.y_val[:50]
    epochs = 1
    batch_size = 8
else:
    print('[안내] GPU 감지 — 전체 데이터로 학습')
    X_tr, y_tr = data.X_train, data.y_train
    X_v,  y_v  = data.X_val,   data.y_val
    epochs = 5
    batch_size = 32

bert = BERTSpamClassifier(
    base_model=MODEL_NAME,
    max_length=128,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=2e-5,
    spam_threshold=0.7,
)

bert.fit(X_tr, y_tr, X_v, y_v)

## Test 세트 평가

In [ ]:
import time
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

X_test_eval = data.X_test[:100] if QUICK_TEST else data.X_test
y_test_eval = data.y_test[:100] if QUICK_TEST else data.y_test

t0 = time.perf_counter()
results = bert.predict(X_test_eval)
elapsed_ms = (time.perf_counter() - t0) * 1000

y_pred = [1 if r.label == 'spam' else 0 for r in results]

print(classification_report(y_test_eval, y_pred, target_names=['ham', 'spam']))
print(f'추론 시간: {elapsed_ms:.1f}ms | 평균 {elapsed_ms/len(X_test_eval):.1f}ms/건')

In [ ]:
cm = confusion_matrix(y_test_eval, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['ham', 'spam'])
disp.plot(cmap='Oranges')
plt.title('혼동 행렬 (BERT)')
plt.tight_layout()
plt.show()

## 모델 저장

In [ ]:
import os
os.makedirs('../artifacts', exist_ok=True)
bert.save('../artifacts/bert_model')
print('저장 완료!')

## 직접 테스트

In [ ]:
test_cases = [
    ('무료 대출 1억 지금 바로 신청', '신용등급 상관없이 당일 대출 가능합니다. 지금 클릭하세요!'),
    ('내일 회의 자료 공유드립니다', '안녕하세요. 내일 오전 10시 회의 자료 첨부드립니다. 검토 부탁드립니다.'),
    ('당신이 선택되었습니다!', '축하합니다! 1등 당첨! 지금 바로 수령하세요. 010-0000-0000'),
]

for subject, body in test_cases:
    text = pre.preprocess(subject, body)
    result = bert.predict_single(text)
    bar = '█' * int(result.spam_probability * 20)
    print(f'[{result.label.upper():4s}] spam={result.spam_probability:.2%} |{bar:<20}| 제목: {subject}')